# **CLASSIFICATION MODEL NOTEBOOK**

## Objectives

* Fit and evaluate a classification model to predict if a hotel booking will be cancelled or not.

## Inputs

* **Raw Dataset:** inputs/datasets/raw/hotel_bookings.csv
* **Data cleaning pipeline** from notebook 4
* Suggested **feature engineering pipeline** from notebook 5

## Outputs

* **Train set** (features and target)
* **Test set** (features and target)
* **Cleaned features dataset** (with amended 'Missing' label on agent)
* Finalised **data cleaning and feature engineering pipelines**
* **Modeling pipeline**
* Feature importance plot

---

# Imports

General imports needed for models and preparing data

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pipelines and custom transformers
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# Data cleaning and feature engineering pipelines 
from feature_engine.imputation import CategoricalImputer, ArbitraryNumberImputer, MeanMedianImputer
from feature_engine.outliers import ArbitraryOutlierCapper
from feature_engine.encoding import OrdinalEncoder, OneHotEncoder, RareLabelEncoder
from feature_engine.creation import CyclicalFeatures, MathFeatures
from feature_engine.transformation import YeoJohnsonTransformer, PowerTransformer
from feature_engine.selection import SmartCorrelatedSelection, DropFeatures

# ML Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

# ML algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

# Load, Pre-Process & Split Data

## Load Data

Load the raw dataset

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
dataset_file = project_root / 'inputs' / 'datasets' / 'raw' / 'hotel_bookings.csv'
df = pd.read_csv(
    dataset_file,
    dtype={
        'agent': 'object',
        'company': 'object',
        'is_repeated_guest': 'object',
    },
)
print('Shape:', df.shape)
df.head(3)

## Pre-Split Data Cleaning

***NOTE:*** *These steps must be done at this stage to avoid data leaking into the test set.*

Drop columns `reservation_status` and `reservation_status_date`

In [ ]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])
print('Shape:', df.shape)
df.head(3)

Aggregate duplicates into single records

In [ ]:
df = df.value_counts(dropna=False).reset_index(name='record_count')
df['is_duplicate'] = (df['record_count'] > 1).astype('int')
print('Shape:', df.shape)
df.head(3)

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('is_canceled', axis=1), df['is_canceled'], test_size=0.2, random_state=0)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train.head(3)

# Define & Apply Pipelines for Data Cleaning & Feature Engineering

Use Data Cleaning pipeline from notebook 4.

Initially, we will use the Feature Engineering pipeline suggested in notebook 5. 

## Data Cleaning Pipeline

Define custom transformer for imputing `is_repeated_guest`

In [ ]:
class RepeatedGuestImputer(BaseEstimator, TransformerMixin):
    def __init__(self, booking_col='previous_bookings_not_canceled', target_col='is_repeated_guest'):
        self.booking_col = booking_col
        self.target_col = target_col

    def fit(self, X, y=None):
        # Set a dummy attribute to signal fitted status (to remove warning)
        self.fitted_ = True
        return self

    def transform(self, X):
        X = X.copy()
        mask_missing = X[self.target_col].isnull()
        X.loc[mask_missing, self.target_col] = (X.loc[mask_missing, self.booking_col] > 0).astype(int)
        return X


Define variables and transformers for using in the pipeline

In [ ]:
# Categorical imputer for 'Missing' label
impute_missing_label_variables = ['country', 'company', 'agent']
imputer_missing_label = CategoricalImputer(
    imputation_method='missing', 
    fill_value='Missing',
    variables=impute_missing_label_variables,
)

# Categorical imputer for mode
impute_mode_variables = [
    'hotel', 'arrival_date_month', 'meal', 'market_segment',
    'distribution_channel', 'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type'
]
imputer_mode = CategoricalImputer(
    imputation_method='frequent',
    variables=impute_mode_variables
)

# Categorical imputer for repeated guest
imputer_repeated_guest = RepeatedGuestImputer()

# Numeric imputer for zero
impute_zero_variables = [
    'children', 'babies', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'required_car_parking_spaces', 'total_of_special_requests'
]
imputer_zero = ArbitraryNumberImputer(
    arbitrary_number=0, 
    variables=impute_zero_variables
)

# Numeric imputer for median
impute_median_variables = ['adr', 'adults']
imputer_median = MeanMedianImputer(
    imputation_method='median',
    variables=impute_median_variables
)

# Outlier capper
max_capping_dict = {
    'lead_time': 600,
    'arrival_date_year': 2017,
    'arrival_date_week_number': 53,
    'arrival_date_day_of_month': 31,
    'stays_in_weekend_nights': 10,
    'stays_in_week_nights': 25,
    'adults': 4,
    'children': 4,
    'babies': 2,
    'previous_cancellations': 10,
    'previous_bookings_not_canceled': 20,
    'booking_changes': 10,
    'days_in_waiting_list': 60,
    'adr': 400,
    'required_car_parking_spaces': 2,
    'total_of_special_requests': 5
}

min_capping_dict = {
    'lead_time': 1,
    'arrival_date_year': 2015,
    'arrival_date_week_number': 1,
    'arrival_date_day_of_month': 1,
    'stays_in_weekend_nights': 0,
    'stays_in_week_nights': 0,
    'adults': 1,
    'children': 0,
    'babies': 0,
    'previous_cancellations': 0,
    'previous_bookings_not_canceled': 0,
    'booking_changes': 0,
    'days_in_waiting_list': 0,
    'adr': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0
}

outlier_capper = ArbitraryOutlierCapper(
    max_capping_dict=max_capping_dict,
    min_capping_dict=min_capping_dict
)

Define data cleaning pipeline

In [ ]:
data_cleaning_pipeline = Pipeline([
    ('imputer_missing_label', imputer_missing_label),
    ('imputer_mode', imputer_mode),
    ('imputer_repeated_guest', imputer_repeated_guest),
    ('imputer_zero', imputer_zero),
    ('imputer_median', imputer_median),
    ('outlier_capper', outlier_capper),
    ('drop_features', DropFeatures(
        features_to_drop=[
            'deposit_type',
            'arrival_date_year',
            'previous_cancellations',
            'previous_bookings_not_canceled',
        ])
    ),
])

**NOTE:** The `drop_features` step has been added because the model will be used to predict whether **future** bookings are likely to cancel or not.
- Removing `arrival_date_year` is necessary because future years will **not have been seen by the model** during training.
- Removing `deposit_type` is advantageous because **decisions around which type of deposit** to require for a particular booking may later be **based on the outcome of the model prediction**.
- `previous_cancellations` and `previous_bookings_not_canceled` may not be available at the time that bookings are made on the PMS, so if the model can predict well without these then it is better to remove them.

Removing these features here (rather than when importing the data) reduces the work needed to add them back in later if required, since the features were already cleaned in earlier steps.

The feature engineering pipeline will need to be amended to reflect this change.

## Feature Engineering Pipeline

Define custom transformers

In [ ]:
class CancellationRatio(BaseEstimator, TransformerMixin):
    """
    Creates cancellation_ratio = previous_cancellations / (previous_total_bookings)
    """

    def __init__(self,
                 cancel_col='previous_cancellations',
                 no_cancel_col='previous_bookings_not_canceled',
                 new_col_name='cancellation_ratio'):
        self.cancel_col = cancel_col
        self.no_cancel_col = no_cancel_col
        self.new_col_name = new_col_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        total_prev = X[self.cancel_col] + X[self.no_cancel_col]
        # Avoid division by zero
        X[self.new_col_name] = np.where(total_prev == 0, 0, X[self.cancel_col] / total_prev)
        return X


class MonthMapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables):
        self.variables = variables
        self.month_map = {
            'January': 1, 'February': 2, 'March': 3, 'April': 4,
            'May': 5, 'June': 6, 'July': 7, 'August': 8,
            'September': 9, 'October': 10, 'November': 11, 'December': 12
        }
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for var in self.variables:
            X[var] = X[var].map(self.month_map)
        return X

Define variables and transformers

In [ ]:
# Ordinal Encoder
binary_ordinal_encoder = OrdinalEncoder(
    encoding_method='arbitrary',
    variables=['hotel', 'is_repeated_guest']
)

# Month Mapper (before cyclical encoding)
month_mapper = MonthMapper(variables=['arrival_date_month'])

# Rare Label Encoders (before one hot encoding)
rare_country_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['country']
)
rare_agent_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['agent']
)
rare_company_encoder = RareLabelEncoder(
    tol=0.01,
    n_categories=1,
    variables=['company']
)

# One-Hot Encoder (after rare label encoders)
one_hot_encoder = OneHotEncoder(
    variables=[
        'meal', 'market_segment', 'distribution_channel', 'reserved_room_type',
        'assigned_room_type', 'customer_type', 'country', 'agent', 'company',
        # 'deposit_type',  # this has been dropped from the data cleaning pipeline
        ],
    drop_last=False
)

# Cyclical Encoders (after Month Mapper)
cyclical_features = CyclicalFeatures(
    variables=['arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month'],
    drop_original=True
)

# Create New Features
create_total_stays = MathFeatures(
    variables=['stays_in_weekend_nights', 'stays_in_week_nights'],
    func='sum',
    new_variables_names=['total_stay_length']
)
create_total_cost = MathFeatures(
    variables=['adr', 'total_stay_length'],
    func='prod',
    new_variables_names=['total_cost']
)

# Remove CancellationRatio - requires previous_cancellations and previous_bookings_not_canceled
# create_cancellation_ratio = CancellationRatio()

# Numeric Transformers
adr_transformer = YeoJohnsonTransformer(variables=['adr'])
lead_time_transformer = PowerTransformer(variables=['lead_time'])

# Smart Correlated Selection
smart_corr_sel = SmartCorrelatedSelection(
    variables=None,
    method='spearman',
    threshold=0.6,
    selection_method='variance'
)

Define feature engineering pipeline

In [ ]:
feature_engineering_pipeline = Pipeline([
    ('binary_ordinal_encoder', binary_ordinal_encoder),
    ('month_mapper', month_mapper),
    ('rare_country_encoder', rare_country_encoder),
    ('rare_agent_encoder', rare_agent_encoder),
    ('rare_company_encoder', rare_company_encoder),
    ('one_hot_encoder', one_hot_encoder),
    ('cyclical_features', cyclical_features),
    ('create_total_stays', create_total_stays),
    ('create_total_cost', create_total_cost),
    # ('create_cancellation_ratio', create_cancellation_ratio),  # Remove - requires previous_cancellations and previous_bookings_not_canceled
    ('adr_transformer', adr_transformer),
    ('lead_time_transformer', lead_time_transformer),
    ('smart_corr_sel', smart_corr_sel),
])

## PipelineDataCleaningAndFeatureEngineering

Combine the previous two pipelines into a single pipeline.

In [ ]:
def PipelineDataCleaningAndFeatureEngineering():
    pipeline_base = Pipeline([
        ("data_cleaning_pipeline", data_cleaning_pipeline),
        ("feature_engineering_pipeline", feature_engineering_pipeline),
    ])

    return pipeline_base

PipelineDataCleaningAndFeatureEngineering()

## Apply PipelineDataCleaningAndFeatureEngineering

Apply `PipelineDataCleaningAndFeatureEngineering` by:
- Fitting pipeline to train dataset
- Transforming both train and test datasets

In [ ]:
pipeline_data_cleaning_feat_eng = PipelineDataCleaningAndFeatureEngineering()
X_train = pipeline_data_cleaning_feat_eng.fit_transform(X_train, y_train)
X_test = pipeline_data_cleaning_feat_eng.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# Define Classification Pipeline & Algorithms

Define classification pipeline

In [ ]:
def PipelineClf(model):
    pipeline_base = Pipeline([
        ("scaler", StandardScaler()),
        # ("feat_selection", SelectFromModel(model)),  # Cannot use this with HistGradientBoostingClassifier
        ("model", model),
    ])

    return pipeline_base

Define classification algorithms and hyperparameters to use in initial search.

***NOTE:*** *We are using the default hyperparameters initially*

In [ ]:
models_quick_search = {
    "LogisticRegression": LogisticRegression(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=0),
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=0),
    "ExtraTreesClassifier": ExtraTreesClassifier(random_state=0),
    "AdaBoostClassifier": AdaBoostClassifier(random_state=0),
    "HistGradientBoostingClassifier": HistGradientBoostingClassifier(random_state=0),
}

params_quick_search = {
    "LogisticRegression": {},
    "XGBClassifier": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "GradientBoostingClassifier": {},
    "ExtraTreesClassifier": {},
    "AdaBoostClassifier": {},
    "HistGradientBoostingClassifier": {},
}

# Define Scorers & Helper Functions

Define all relevant scoring metrics that can be used when evaluating model performance.

Since we are prioritising a recall above 0.8, we will use recall_scorer initially.

In [ ]:
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score

recall_scorer = make_scorer(recall_score, pos_label=1)
precision_scorer = make_scorer(precision_score, pos_label=1)
f1_scorer = make_scorer(f1_score, pos_label=1)

Helper for finding best model and hyperparameters (written by Code Institute)

In [ ]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineClf(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

Adapted from a function written by Code Institute
- Includes extra summary tables to more easily compare the model performance on testing and training data.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def target_class_summary(X_train, y_train, X_test, y_test, pipeline, label_map, target_class):
    """
    Generate a summary of precision, recall and F1-score for a specific target class 
    on both training and test datasets.
    """

    # Model performance on training dataset
    y = y_train
    prediction = pipeline.predict(X_train)

    train_report = classification_report(
        y, prediction, target_names=label_map, output_dict=True
    )
    train_precision = train_report[target_class]["precision"]
    train_recall = train_report[target_class]["recall"]
    train_f1 = train_report[target_class]["f1-score"]

    # Model performance on testing dataset
    y = y_test
    prediction = pipeline.predict(X_test)

    test_report = classification_report(
        y, prediction, target_names=label_map, output_dict=True
    )
    test_precision = test_report[target_class]["precision"]
    test_recall = test_report[target_class]["recall"]
    test_f1 = test_report[target_class]["f1-score"]

    results = {
        "Dataset": ['Train', 'Test'],
        "Precision": [f"{train_precision:.2f}", f"{test_precision:.2f} ({(test_precision - train_precision):.2f})"],
        "Recall": [f"{train_recall:.2f}", f"{test_recall:.2f} ({(test_recall - train_recall):.2f})"],
        "F1-Score": [f"{train_f1:.2f}", f"{test_f1:.2f} ({(test_f1 - train_f1):.2f})"],
    }

    overview =  pd.DataFrame(results).set_index('Dataset')
    overview.index.name = None
    display(overview)


def confusion_matrix_and_report(X, y, pipeline, label_map):
    """
    Print a confusion matrix and classification report for predictions on the given dataset.
    """
    prediction = pipeline.predict(X)

    print("---  Confusion Matrix  ---")
    cm = confusion_matrix(y_true=y, y_pred=prediction)
    print(
        pd.DataFrame(
            cm,
            index=["Actual " + sub for sub in label_map],
            columns=["Predicted " + sub for sub in label_map],
        )
    )
    print("\n")

    print("---  Classification Report  ---")
    print(classification_report(y, prediction, target_names=label_map), "\n")


def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map, target_class=None, summary_only=False):
    """
    Evaluate classification performance on train and test datasets, 
    with optional focus on a specific target class.
    """

    # Show Summary
    if target_class:
        print(f'#### Summary on "{target_class}" class ####')
        target_class_summary(X_train, y_train, X_test, y_test, pipeline, label_map, target_class)
        
    else:
        print('No target class specified\n')
    
    # Show Drilldown
    if not summary_only:
        print("#### Train Set #### \n")
        confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

        print("#### Test Set ####\n")
        confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

# Target Imbalance

## Approaches for Handling Target Imbalance

View distribution of target values in train set

In [ ]:
y_train.value_counts().plot(kind='bar', title='Train Set Target Distribution')
plt.show()

We will test the performance of the various models using 3 different approaches:
1. Keep Imbalance
2. Oversample (SMOTE)
3. Undersample

Next we will prepare edited training data to use for each approach.

## Prepare Data for Each Approach

### 1. Keep Imbalance

We can just use X_train and y_train unedited for this test.

**NO ACTION REQUIRED**

### 2. OverSample (SMOTE)

Fit SMOTE to training data and save oversampled data as `X_train_over` and `y_train_over`

In [ ]:
from imblearn.over_sampling import SMOTE

oversample = SMOTE(sampling_strategy='minority', random_state=0)
X_train_over, y_train_over = oversample.fit_resample(X_train, y_train)

print('Observations in original data:', X_train.shape)
print('Observations in oversampled data:', X_train_over.shape)

Check target distribution of oversampled data

In [ ]:
y_train_over.value_counts().plot(kind='bar', title='Train Set Target Distribution')
plt.show()

### 3. Undersample

Fit undersampler to training data and save undersampled data as `X_train_under` and `y_train_under`

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

undersample = RandomUnderSampler(sampling_strategy='majority', random_state=0)
X_train_under, y_train_under = undersample.fit_resample(X_train, y_train)

print('Observations in original data:', X_train.shape)
print('Observations in undersampled data:', X_train_under.shape)

Check target distribution of undersampled data

In [ ]:
y_train_under.value_counts().plot(kind='bar', title='Train Set Target Distribution')
plt.show()

# Find Best Algorithm & Target Imbalance Approach

## Approach 1: Train Models on Imbalanced Data

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- use recall as performance metric

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train, y_train, scoring=recall_scorer, n_jobs=-1, cv=5)

Show Grid Search results

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
display(grid_search_summary.head(5))

top_n = min(5, len(grid_search_summary))
for i in range(top_n):
    # Best Model
    model = grid_search_summary.iloc[i,0]
    print(f'\nModel: {model}')

    # Assign pipeline
    pipeline_clf = grid_search_pipelines[model].best_estimator_

    # Show summary
    clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map=['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only = True
                )


## Approach 2: Train Models on Oversampled Data

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- use recall as performance metric

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train_over, y_train_over, scoring=recall_scorer, n_jobs=-1, cv=5)

Show Grid Search results

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
display(grid_search_summary.head(5))

top_n = min(5, len(grid_search_summary))
for i in range(top_n):
    # Best Model
    model = grid_search_summary.iloc[i,0]
    print(f'\nModel: {model}')

    # Assign pipeline
    pipeline_clf = grid_search_pipelines[model].best_estimator_

    # Show summary
    clf_performance(X_train=X_train_over, y_train=y_train_over,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map=['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only = True
                )


## Approach 3: Train Models on Undersampled Data

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search
- use recall as performance metric

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train_under, y_train_under, scoring=recall_scorer, n_jobs=-1, cv=5)

Show Grid Search results

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
display(grid_search_summary.head(5))

top_n = min(5, len(grid_search_summary))
for i in range(top_n):
    # Best Model
    model = grid_search_summary.iloc[i,0]
    print(f'\nModel: {model}')

    # Assign pipeline
    pipeline_clf = grid_search_pipelines[model].best_estimator_

    # Show summary
    clf_performance(X_train=X_train_under, y_train=y_train_under,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map=['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only = True
                )


## Analysis of Approaches (1 to 3)

When considering the best performing models for each dataset (by 'Cancel' category), here are the findings:
- **Unbalanced:**
  - lowest recall (~0.63) but highest precision (~0.73) - expected since 'Cancel' category is underrepresented
  - F1 scores ~0.68
  - `XGBClassifier` and `HistGradientBoostingClassifier` had the strongest performance
  - `ExtraTreesClassifier` and `RandomForestClassifier` performed well but overfitted

- **SMOTE:**
  - recall was slightly higher (~0.66) but precision slightly lower (~0.71)
  - F1 scores ~0.62
  - `GradientBoostingClassifier` performed the best for recall

- **Under-Sample:**
  - recall was higher (~0.85) but precision was lower (~0.60)
  - F1 scores ~0.70
  - `XGBClassifier` and `HistGradientBoostingClassifier` performed well
  - `ExtraTreesClassifier` and `RandomForestClassifier` performed well but overfitted

**CONCLUSIONS**
- Oversampling with SMOTE had a negative impact on the F1 score (due to the lower precision) so will not be used.
- `XGBClassifier` and `HistGradientBoostingClassifier` had the strongest performance on the imbalanced and undersampled datasets.
- Since the xgboost library is very large, it is not possible to deploy this model with Heroku. Therefore, `HistGradientBoostingClassifier` is the model that will be used moving forward.

## Approach 4: Train `HistGradientBoostingClassifier` on Imbalanced Data using `class_weight`

The `HistGradientBoostingClassifier` model has a parameter called `class_weight` which can be used to address target imbalance. Therefore, this approach will also be investigated.

Define model and parameters

In [ ]:
models_quick_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0),
}

params_quick_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': ['balanced'],
    },
}

Apply `HyperparameterOptimizationSearch` to carry out an SKLearn Grid Search using class_weight as the only defined parameter.

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train, y_train, scoring=recall_scorer, n_jobs=-1, cv=5)

Grid Search Summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary

Evaluate model performance

In [ ]:
clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline_clf,
                label_map=['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only=False
                )

## Conclusion and Next Steps

Approach 4 gave the strongest performance of all of the approaches tested.

Using `HistGradientBoostingClassifier` with `class_weight="balanced"` as the only parameter was enough to satisfy the business requirements!
- Recall on 'Cancel' is at least 0.8
- Precision on 'Cancel' is at least 0.6

A hyperparameter optimisation search will now be conducted using `HistGradientBoostingClassifier` to see if any improvements can be made to the model. Since the precision value is only just above the required threshold, it may be necessary to use precision or f1-score as the scoring metric.

# Hyperparameter Optimisation Search

## 1. Quick search Recall

Define model and parameters to search.
- Since the default model is not overfitting, these parameters will allow for deeper searches

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': ['balanced'],
        'model__learning_rate': [0.05, 0.1, 0.2],  # default is 0.1
        'model__max_iter': [50, 100, 200],  # default is 100
        'model__max_leaf_nodes': [20, 31, 50],  # default is 31
        'model__max_depth': [5, 10, None],  # default is None
        'model__max_bins': [128],  # to make search faster
        'model__early_stopping': [True],
        'model__scoring': ['loss']  # loss is more stable than f1 or recall
    }
}

Fit models
- use recall scorer first
- if precision falls below 0.6, can try precision or F1 scorer instead

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring=recall_scorer, n_jobs=-1, cv=5)

Grid Search Summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(10)

Evaluate performance of best model

In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline_clf,
    label_map=['No Cancel', 'Cancel'],
    target_class='Cancel',
    summary_only=False
)

**CONCLUSION:**
Although the recall improved, the precision on 'Cancel' dropped below the threshold of 0.6. Using precision as the scoring metric instead may yield better results.

## 2. Quick search Precision (improve precision)

The same parameters will be tried again using precision as the scoring metric.

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

# These parameters took about 7 mins to search (405 fits)
params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': ['balanced'],
        'model__learning_rate': [0.05, 0.1, 0.2],  # default is 0.1
        'model__max_iter': [50, 100, 200],  # default is 100
        'model__max_leaf_nodes': [20, 31, 50],  # default is 31
        'model__max_depth': [5, 10, None],  # default is None
        'model__max_bins': [128],  # to make search faster
        'model__early_stopping': [True],
        'model__scoring': ['loss']  # loss is more stable than f1 or recall
    }
}

Fit models using precision scorer instead

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring=precision_scorer, n_jobs=-1, cv=5)

Grid Search Summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(10)

Evaluate performance of best model

In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline_clf,
    label_map=['No Cancel', 'Cancel'],
    target_class='Cancel',
    summary_only=False
)

**CONCLUSION:**
This gave a more balanced model with good Recall and Precision. However, the model did not generalise as well (overfitted slghtly). Next we will try to reduce overfitting.

## 3. Narrow search Precision (prevent overfitting)

Overfitting was likely caused by the following:
- `max_iter=200` - too high
- `max_leaf_nodes=50` - too high
- `max_depth=None` - no limit (although this is the default)

Therefore, these values will be reduced to see if the model generalises better.

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': ['balanced'],
        
        # these have been reduced / limited to prevent overfitting
        'model__max_iter': [100],
        'model__max_leaf_nodes': [31],
        'model__max_depth': [5],

        # these have been kept the same for a fair comparison
        'model__learning_rate': [0.2],
        'model__max_bins': [128],
        'model__early_stopping': [True],
        'model__scoring': ['loss']
    }
}

Fit models using precision scorer

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring=precision_scorer, n_jobs=-1, cv=5)

Grid Search Summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(30)

Evaluate performance of best model

In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline_clf,
    label_map=['No Cancel', 'Cancel'],
    target_class='Cancel',
    summary_only=False
)

**CONCLUSION:**
These parameters maintained a good balance between recall and precision without overfitting. Next, we will 'lock-in' these values to maintain the good f1 score and see if any improvements can be made by changing a few other parameters.

## 4. Lock-in and Extend Search Precision (optimise further) 

Some additional parameters are added

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': ['balanced'],
        
        # locked-in from earlier to maintain good F1-score
        'model__learning_rate': [0.2],
        'model__max_iter': [100],  # 150
        'model__max_leaf_nodes': [31],  #31
        'model__max_depth': [5],  # 10
        'model__early_stopping': [True],
        'model__scoring': ['loss'],

        # New parameters to try
        'model__min_samples_leaf': [5, 20, 50],  # default is 20
        'model__l2_regularization': [0.0, 0.1, 1],  # default is 0
        'model__max_bins': [128, 255],  # default is 255 and also max value
    }
}

Fit models using f1 scorer

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring=precision_scorer, n_jobs=-1, cv=5)

Grid Search Summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(30)

Evaluate performance of best model

In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline_clf,
    label_map=['No Cancel', 'Cancel'],
    target_class='Cancel',
    summary_only=False
)

**CONCLUSION:**
The model maintained good scores for both recall and precision on 'Cancel'.

## Overall Conclusion

The hyperparameter optimisation search confirmed that when `class_weight='balanced'` is used, the default values for all other parameters are already very nearly optimised for this data, and departing too far from these values has a negative effect on the F1-score.

The best parameters are:

In [ ]:
best_parameters

# Prepare Pipeline for Deployment

To avoid having to do the hyperparameter optimisation searches to get the best model and pipeline, let's fit the best model again here.

## Fit Best Model (using all features)

These are the best model parameters found through the hyperparameter optimisation search

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': ['balanced'],
        'model__early_stopping': [True],
        'model__l2_regularization': [0.1],
        'model__learning_rate': [0.2],
        'model__max_bins': [255],
        'model__max_depth': [5],
        'model__max_iter': [100],
        'model__max_leaf_nodes': [31],
        'model__min_samples_leaf': [5],
        'model__scoring': ['loss']
    }
}

Fit the model using the best parameters

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring=precision_scorer, n_jobs=-1, cv=5)

Grid search summary allows us to access `pipeline_clf` for the next step

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(5)

## Assess Feature Importance

Since `HistGradientBoostingClassifier` doesn't have an attribute like `feature_importances_`, we will use `permutation_importance` instead.

In [ ]:
from sklearn.inspection import permutation_importance

# Run permutation importance on the best pipeline
result = permutation_importance(
    pipeline_clf, 
    X_test, y_test, 
    n_repeats=10, 
    random_state=0, 
    n_jobs=-1,
    scoring='accuracy'
)

# Feature Names
feature_names = list(X_train.columns)

# Build DataFrame
df_feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": result.importances_mean
    })
    .sort_values(by="Importance", ascending=False)
)

plt.figure(figsize=(12, 6))
df_feature_importance.plot(
    kind="bar", x="Feature", y="Importance", legend=False,
    title="Permutation Feature Importance", figsize=(12, 6)
)
plt.ylabel("Mean importance decrease (permutation)")
plt.tight_layout()
plt.show()


Get the best features

In [ ]:
top_n = 12
df_best_features = df_feature_importance.head(top_n)

# Re-assign order for later use
best_features = df_best_features["Feature"].to_list()

# --- Print and Plot ---
print(f"* These are the {len(best_features)} most important features in descending order.\n")
print(best_features)

plt.figure(figsize=(12, 6))
df_best_features.plot(
    kind="bar", x="Feature", y="Importance", legend=False,
    title="Permutation Feature Importance", figsize=(12, 6)
)
plt.ylabel("Mean importance decrease (permutation)")
plt.tight_layout()
plt.show()

## Evaluate Model Performance Using Only Best Features

Fit model using only best features
- may need to try tweaking the threshold for selecting best features in the previous section.

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train[best_features], y_train, scoring=recall_scorer, n_jobs=-1, cv=5)

Grid search summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(5)

In [ ]:
clf_performance(X_train=X_train[best_features], y_train=y_train,
                X_test=X_test[best_features], y_test=y_test,
                pipeline=pipeline_clf,
                label_map=['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only=False
                )

**CONCLUSION:** The model keeps a good F1 score without overfitting using only the top features. However, the precision has dropped below 0.6.

We can adjust the balance between recall and precision using class weights.

## Adjust Precision / Recall Balance

Get class_weight which is slightly off-balanced to favour precision 

In [ ]:
num_negative = (y_train == 0).sum()
num_positive = (y_train == 1).sum()
n_samples = len(y_train)

# Get balanced weights
w0 = 1
w1 = num_negative / num_positive
class_weight = {0: w0, 1: w1}
print('Class weights:', class_weight)

# Precision tilt (favor negatives slightly more)
class_weight_precision = {0: w0 * 1.1, 1: w1}
print('Precision tilt:', class_weight_precision)


Here are the best model parameters using adjusted class_weight

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': [class_weight_precision],  # bias towards precision
        'model__early_stopping': [True],
        'model__l2_regularization': [0.1],
        'model__learning_rate': [0.2],
        'model__max_bins': [255],
        'model__max_depth': [5],
        'model__max_iter': [100],
        'model__max_leaf_nodes': [31],
        'model__min_samples_leaf': [5],
        'model__scoring': ['loss']
    }
}

Fit the model using only the most important features

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train[best_features], y_train, scoring=precision_scorer, n_jobs=-1, cv=5)

Grid search summary allows us to access `pipeline_clf` for the next step

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(5)

In [ ]:
clf_performance(X_train=X_train[best_features], y_train=y_train,
                X_test=X_test[best_features], y_test=y_test,
                pipeline=pipeline_clf,
                label_map=['No Cancel', 'Cancel'],
                target_class='Cancel',
                summary_only=False
                )

**CONCLUSION:** This model is sufficient for meeting the business requirements and requires only the following features:

In [ ]:
best_features

## Streamline Pipeline

### Define `original_best_features`

Some of the features above were derived from other features
- define `original_best_features`

In [ ]:
original_best_features = [
    'country',
    'lead_time',
    'total_of_special_requests',
    'agent',
    'required_car_parking_spaces',
    'customer_type',
    'stays_in_weekend_nights',  # needed to calculate total cost
    'stays_in_week_nights',  # needed to calculate total cost
    'booking_changes',
    'record_count',
    'arrival_date_week_number',
    'adr',
]

### Load Raw Dataset

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent
dataset_file = project_root / 'inputs' / 'datasets' / 'raw' / 'hotel_bookings.csv'
df = pd.read_csv(
    dataset_file,
    dtype={
        'agent': 'object',
        'company': 'object',
        'is_repeated_guest': 'object',
    },
)
print('Shape:', df.shape)
df.head(3)

### Pre-Split Data Cleaning

Drop `reservation_status` and `reservation_status_date`

In [ ]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])
print('Shape:', df.shape)
df.head(3)

Aggregate duplicates into single records

In [ ]:
df = df.value_counts(dropna=False).reset_index(name='record_count')
df['is_duplicate'] = (df['record_count'] > 1).astype('int')
print('Shape:', df.shape)
df.head(3)

Now drop all features not in `original_best_features`
- we need to keep the target variable too

In [ ]:
df = df[original_best_features + ['is_canceled']]
print('Shape:', df.shape)
df.head(3)

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('is_canceled', axis=1), df['is_canceled'], test_size=0.2, random_state=0)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train.head(3)

Make a copy of X_train before processing for use with the cancellation predictor later

In [ ]:
uncleaned_features = X_train.copy()

### Streamline Data Cleaning Pipeline

Define variables and transformers

In [ ]:
# Categorical imputer for 'Missing' label
# - Removed company
imputer_missing_label = CategoricalImputer(
    imputation_method='missing', 
    fill_value='Missing',
    variables=['country', 'agent'],
)

# Categorical imputer for mode
# - Removed hotel, meal, distribution_channel, reserved_room_type, assigned_room_type,
#       deposit_type, arrival_date_month, arrival_date_month, market_segment
imputer_mode = CategoricalImputer(
    imputation_method='frequent',
    variables=['customer_type']
)

# RepeatedGuestImputer no longer needed

# Numeric imputer for zero
# - Removed children, babies, previous_cancellations, previous_bookings_not_canceled
imputer_zero = ArbitraryNumberImputer(
    arbitrary_number=0, 
    variables=[
        'booking_changes',
        'required_car_parking_spaces',
        'total_of_special_requests'
    ]
)

# Numeric imputer for median
# - Removed adults
imputer_median = MeanMedianImputer(
    imputation_method='median',
    variables=['adr']
)

# Outlier capper
max_capping_dict = {
    'lead_time': 600,
    'arrival_date_week_number': 53,
    'stays_in_weekend_nights': 10,
    'stays_in_week_nights': 25,
    'booking_changes': 10,
    'adr': 400,
    'required_car_parking_spaces': 2,
    'total_of_special_requests': 5
}

min_capping_dict = {
    'lead_time': 1,
    'arrival_date_week_number': 1,
    'stays_in_weekend_nights': 0,
    'stays_in_week_nights': 0,
    'booking_changes': 0,
    'adr': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0
}

# - Removed following from both lists:
#     previous_cancellations, previous_bookings_not_canceled, arrival_date_year, 
#     arrival_date_day_of_month, adults, children, babies, days_in_waiting_list
outlier_capper = ArbitraryOutlierCapper(
    max_capping_dict=max_capping_dict,
    min_capping_dict=min_capping_dict
)

Define data cleaning pipeline

In [ ]:
data_cleaning_pipeline = Pipeline([
    ('imputer_missing_label', imputer_missing_label),
    ('imputer_mode', imputer_mode),
    # imputer_repeated_guest no longer needed
    ('imputer_zero', imputer_zero),
    ('imputer_median', imputer_median),
    ('outlier_capper', outlier_capper),
    # drop_features no longer needed
])

### Streamline Feature Engineering Pipeline

Define variables and transformers

In [ ]:
# binary_ordinal_encoder no longer needed

# Month Mapper no longer needed

# Rare Label Encoders (before one hot encoding)
rare_country_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['country']
)
rare_agent_encoder = RareLabelEncoder(
    tol=0.005,
    n_categories=1,
    variables=['agent'],
)

# RareLabelEncoder on company no longer needed

# One-Hot Encoder (after rare label encoders)
# - Removed meal, distribution_channel, reserved_room_type, assigned_room_type,
#      company, deposit_type, market_segment
one_hot_encoder = OneHotEncoder(
    variables=['customer_type', 'country', 'agent'],
    drop_last=False
)

# Cyclical Encoders (after Month Mapper)
# - Removed arrival_date_day_of_month, 'arrival_date_month', 
cyclical_features = CyclicalFeatures(
    variables=['arrival_date_week_number'],
    drop_original=True
)

# Create New Features
# create_total_stays needed to create total_cost
create_total_stays = MathFeatures(
    variables=['stays_in_weekend_nights', 'stays_in_week_nights'],
    func='sum',
    new_variables_names=['total_stay_length']
)
create_total_cost = MathFeatures(
    variables=['adr', 'total_stay_length'],
    func='prod',
    new_variables_names=['total_cost']
)

# create_cancellation_ratio no longer needed

# Numeric Transformers
adr_transformer = YeoJohnsonTransformer(variables=['adr'])
lead_time_transformer = PowerTransformer(variables=['lead_time'])

# Smart Correlated Selection no longer needed

Define feature engineering pipeline

In [ ]:
feature_engineering_pipeline = Pipeline([
    # binary_ordinal_encoder removed
    # month_mapper removed
    ('rare_country_encoder', rare_country_encoder),
    ('rare_agent_encoder', rare_agent_encoder),
    # rare_company_encoder removed
    ('one_hot_encoder', one_hot_encoder),
    ('cyclical_features', cyclical_features),
    ('create_total_stays', create_total_stays),
    ('create_total_cost', create_total_cost),
    # create_cancellation_ratio removed
    ('adr_transformer', adr_transformer),
    ('lead_time_transformer', lead_time_transformer),
    # smart_corr_sel removed
])

### Create `DataCleaningAndFeatureEngineering` Pipeline and Apply 

Combine previous two pipelines

In [ ]:
def PipelineDataCleaningAndFeatureEngineering():
    pipeline_base = Pipeline([
        ("data_cleaning_pipeline", data_cleaning_pipeline),
        ("feature_engineering_pipeline", feature_engineering_pipeline),
    ])

    return pipeline_base

PipelineDataCleaningAndFeatureEngineering()

Apply `PipelineDataCleaningAndFeatureEngineering` by:
- Fitting pipeline to train dataset
- Transforming both train and test datasets

In [ ]:
pipeline_data_cleaning_feat_eng = PipelineDataCleaningAndFeatureEngineering()
X_train = pipeline_data_cleaning_feat_eng.fit_transform(X_train, y_train)
X_test = pipeline_data_cleaning_feat_eng.transform(X_test)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

### Define Classification Pipeline

In [ ]:
def PipelineClf(model):
    pipeline_base = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model),
    ])

    return pipeline_base

### Evaluate Performance Using `original_best_features`

Here are the best model parameters for the model using only the most important features

In [ ]:
models_search = {
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier(random_state=0)
}

params_search = {
    'HistGradientBoostingClassifier': {
        'model__class_weight': [class_weight_precision],  # bias towards precision
        'model__early_stopping': [True],
        'model__l2_regularization': [0.1],
        'model__learning_rate': [0.2],
        'model__max_bins': [255],
        'model__max_depth': [5],
        'model__max_iter': [100],
        'model__max_leaf_nodes': [31],
        'model__min_samples_leaf': [5],
        'model__scoring': ['loss']
    }
}

Fit the model using the best parameters (with reduced features)

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring=recall_scorer, n_jobs=-1, cv=5)

Grid search summary

In [ ]:
# Summary results
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

# Best Model
best_model = grid_search_summary.iloc[0,0]
print(f'Best model: {best_model}')

# Best Parameters
best_parameters = grid_search_pipelines[best_model].best_params_
print(f'Best parameters: {best_parameters}')

# Assign these to classification pipeline
pipeline_clf = grid_search_pipelines[best_model].best_estimator_

grid_search_summary.head(5)

Evaluate performance of best model

In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline_clf,
    label_map=['No Cancel', 'Cancel'],
    target_class='Cancel',
    summary_only=False
)

### Check Feature Importance

There should be fewer features than when we ran this before

In [ ]:
# Run permutation importance on the best pipeline
result = permutation_importance(
    pipeline_clf, 
    X_test, y_test, 
    n_repeats=10, 
    random_state=0, 
    n_jobs=-1
)

# Feature Names
feature_names = list(X_train.columns)

# Build DataFrame
df_feature_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": result.importances_mean
    })
    .sort_values(by="Importance", ascending=False)
)

plt.figure(figsize=(12, 6))
df_feature_importance.plot(
    kind="bar", x="Feature", y="Importance", legend=False,
    title="Permutation Feature Importance", figsize=(12, 6)
)
plt.ylabel("Mean importance decrease (permutation)")
plt.tight_layout()
plt.show()


Get the best features again

In [ ]:
top_n = 12
df_best_features = df_feature_importance.head(top_n)

# Re-assign order for later use
best_features = df_best_features["Feature"].to_list()

# --- Print and Plot ---
print(f"* These are the {len(best_features)} most important features in descending order.\n")
print(best_features)

plt.figure(figsize=(12, 6))
df_best_features.plot(
    kind="bar", x="Feature", y="Importance", legend=False,
    title="Permutation Feature Importance", figsize=(12, 6)
)
plt.ylabel("Mean importance decrease (permutation)")
plt.tight_layout()
plt.show()

# Push files to Repo

## Define Version and Useful Functions

Set version number for outputs directory

In [ ]:
version = 'v1'

Useful functions

In [ ]:
def ensure_output_directory_exists(dirpath):
    if not dirpath.is_dir():
        dirpath.mkdir(parents=True, exist_ok=False)
        print(
            'NOTICE: The following directory did not already exist so has been created:\n',
            dirpath,
            '\n\nPlease check that the version specified above is correct.'
        )
    else:
        print('Found outputs version directory:\n', dirpath)


def save_dataframe_as_csv(df, filepath, index=True):
    # Check file does not already exist
    if filepath.exists():
        raise FileExistsError(
            f'File already exists at: {filepath}\n'
            'Please delete the old file if no longer needed and run this cell again.'
        )
    
    # Save file as csv file
    df.to_csv(filepath, index=index)

    # Confirmation message
    print(f'SUCCESS: file saved at {filepath}')


## Ensure Output Directories Exist 

Check outputs **version** directory already exists. Otherwise, create it.
- ***NOTE:*** *It should have been created in notebook 2*

In [ ]:
outputs_v_dir = project_root / 'outputs' / version
ensure_output_directory_exists(outputs_v_dir)

Check outputs directory for **cleaned datasets** already exists. Otherwise, create it.
- ***NOTE:*** *It should have been created in notebook 2*

In [ ]:
dataset_cleaned_dir = outputs_v_dir / 'datasets' / 'cleaned'
ensure_output_directory_exists(dataset_cleaned_dir)

Check outputs **images** directory already exists. Otherwise, create it.
- ***NOTE:*** *It should have been created in notebook 2*

In [ ]:
images_dir = outputs_v_dir / 'images'
ensure_output_directory_exists(images_dir)

Create outputs **pipelines** directory.
- ***NOTE:*** *This directory is not created in earlier notebooks.*

In [ ]:
pipelines_dir = outputs_v_dir / 'ml_pipelines'
ensure_output_directory_exists(pipelines_dir)

## Save Dataframes

### Train and Test Sets

- ***NOTE:*** *The features in `X_train` and `X_test` have already been processed using the DataCleaningAndFeatureEngineering pipeline*

In [ ]:
filepath = dataset_cleaned_dir / 'X_train.csv'
save_dataframe_as_csv(X_train, filepath, index=False)

In [ ]:
filepath = dataset_cleaned_dir / 'y_train.csv'
save_dataframe_as_csv(y_train, filepath, index=False)

In [ ]:
filepath = dataset_cleaned_dir / 'X_test.csv'
save_dataframe_as_csv(X_test, filepath, index=False)

In [ ]:
filepath = dataset_cleaned_dir / 'y_test.csv'
save_dataframe_as_csv(y_test, filepath, index=False)

### Cleaned Features (for use with predictor)

`x_train_unprocessed` is a copy of X_train before it has been transformed by the data cleaning and feature engineering pipelines. We will apply the data cleaning pipeline to this as safe data to use with the cancellation predictor on the dashboard.

In [ ]:
cleaned_features = data_cleaning_pipeline.fit_transform(uncleaned_features)
cleaned_features.head(5)

For the purposes of the cancellation predictor, it would be clearer if the 'Missing' label for `agent` and `country` was renamed to 'None / Unknown'.

In [ ]:
cleaned_features['agent'] = cleaned_features['agent'].replace('Missing', 'None / Unknown')
cleaned_features['country'] = cleaned_features['country'].replace('Missing', 'None / Unknown')
cleaned_features.head(5)

Save

In [ ]:
filepath = dataset_cleaned_dir / 'cleaned_features.csv'
save_dataframe_as_csv(cleaned_features, filepath, index=False)

## Save Fitted Pipelines

We will save both fitted pipelines: `pipeline_data_cleaning_feat_eng` and `pipeline_clf`:
- Both are needed to predict Live Data.
- To predict on Train and Test Sets only `pipeline_clf` is used (since the data is already processed).

In [ ]:
import joblib

### Fitted Pipeline for Data Cleaning and Feature Engineering

View pipeline

In [ ]:
pipeline_data_cleaning_feat_eng

Save

In [ ]:
filepath = pipelines_dir / 'pipeline_data_cleaning_feat_eng.pkl'
joblib.dump(value=pipeline_data_cleaning_feat_eng, filename= filepath)

### Fitted Classification Pipeline

View pipeline

In [ ]:
pipeline_clf

Save

In [ ]:
filepath = pipelines_dir / 'pipeline_clf.pkl'
joblib.dump(value=pipeline_clf, filename= filepath)

## Save Images

Image showing the most important features that were used to train the model

In [ ]:
top_n = 12
df_best_features = df_feature_importance.head(top_n)

# Re-assign order for later use
best_features = df_best_features["Feature"].to_list()

# --- Print and Plot ---
print(f"* These are the {len(best_features)} most important features in descending order.\n")
print(best_features)

plt.figure(figsize=(12, 6))
df_best_features.plot(
    kind="bar", x="Feature", y="Importance", legend=False,
    title="Permutation Feature Importance", figsize=(12, 6)
)
plt.ylabel("Mean importance decrease (permutation)")
plt.tight_layout()

# Save image
filepath = images_dir / 'feature_importance.png'
plt.savefig(filepath, bbox_inches='tight')
print(f'SUCCESS: Image saved at {filepath}')

plt.show()